In [3]:
"""
SOFA HRTF Frontal Looming Sound Simulator

This script creates binaural audio files simulating a sound source approaching the listener
from the front (0° azimuth), using the Fabian HRTF models for directional cues and a 
physically-based distance model inspired by the EPFL LEMA approach.

Multiple noise types are supported: pink, white, brown, blue, and purple.
"""
import numpy as np
import scipy.signal as signal
import soundfile as sf
import matplotlib.pyplot as plt
import os
import time
import datetime

# First, ensure required libraries are installed
try:
    import sofar
    SOFA_AVAILABLE = True
except ImportError:
    print("Warning: sofar library not found. Please install with: pip install sofar")
    SOFA_AVAILABLE = False

#############################################################
# CONTROL PARAMETERS - ADJUST THESE TO CHANGE THE SIMULATION
#############################################################

# Spatial parameters
INITIAL_DISTANCE = 2.0    # Starting distance in meters
FINAL_DISTANCE = 0.5      # Final distance in meters (increased to avoid extreme effects)
AZIMUTH_ANGLE = 0         # Fixed to 0 degrees for frontal looming
CONSTANT_ELEVATION = 0    # Fixed elevation (ear level)
DISTANCE_FACTOR = 40      # Attenuation factor (higher = stronger distance effect)
HEAD_DIAMETER = 0.18      # Diameter of the head in meters (natural human head size)

# Audio parameters
VOLUME_LEVEL = 1          # Output volume factor (0.0 to 1.0, higher = louder)

# Time parameters
DURATION = 3.0            # Duration of movement in seconds
SAMPLE_RATE = 44100       # Audio sample rate in Hz
SPEED_OF_SOUND = 343.0    # Speed of sound in m/s

# Noise types to generate
NOISE_TYPES = ['pink', 'white', 'brown', 'blue']  # The most "natural" noise types

# Use only HRIR files for maximum binaural effect
sofa_files = [
    r"C:\Users\cogpsy-vrlab\Downloads\HRIRs_neutral_head_orientation_v4\SOFA\FABIAN_HRIR_measured_HATO_0.sofa"
]

# Create output directory
output_dir = r"C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles"
os.makedirs(output_dir, exist_ok=True)

def generate_noise(noise_type, duration, fs):
    """Generate different types of noise"""
    # Number of samples
    num_samples = int(duration * fs)
    
    # Start with white noise
    np.random.seed(42)  # For reproducibility
    white_noise = np.random.randn(num_samples)
    
    if noise_type == 'white':
        # White noise is already generated
        noise = white_noise
    elif noise_type == 'pink':
        # Pink noise using filter method
        b = np.array([0.049922035, -0.095993537, 0.050612699, -0.004408786])
        a = np.array([1.0, -2.494956002, 2.017265875, -0.522189400])
        noise = signal.lfilter(b, a, white_noise)
    elif noise_type == 'brown':
        # Brown noise (integrated white noise)
        # Create a simple low pass filter
        b = [1.0]
        a = [1.0, -0.98]
        noise = signal.lfilter(b, a, white_noise)
    elif noise_type == 'blue':
        # Blue noise (high frequency emphasis)
        # Create a simple high pass filter
        b = [1.0, -1.0]
        a = [1.0, -0.98]
        noise = signal.lfilter(b, a, white_noise)
    elif noise_type == 'purple':
        # Purple noise (high frequency emphasis, steeper than blue)
        # Apply filter twice for steeper curve
        b = [1.0, -1.0]
        a = [1.0, -0.98]
        noise = signal.lfilter(b, a, white_noise)
        noise = signal.lfilter(b, a, noise)
    else:
        # Default to white noise if type not recognized
        print(f"Noise type '{noise_type}' not recognized. Using white noise instead.")
        noise = white_noise
    
    # Normalize
    noise = noise / np.max(np.abs(noise)) * 0.8
    
    return noise

def load_sofa_file(sofa_path):
    """Load HRTF data from SOFA file"""
    try:
        print(f"Loading SOFA file: {os.path.basename(sofa_path)}")
        
        # Load the SOFA file
        hrtf = sofar.read_sofa(sofa_path)
        
        # Display basic SOFA info
        print(f"SOFA convention: {hrtf.GLOBAL_SOFAConventions} v{hrtf.GLOBAL_SOFAConventionsVersion}")
        
        # Get dimensions
        dimensions = {}
        for dim_name in ['M', 'R', 'N']:
            try:
                dimensions[dim_name] = hrtf.get_dimension(dim_name)
                print(f"{dim_name} = {dimensions[dim_name]}")
            except:
                dimensions[dim_name] = None
                print(f"{dim_name} = Unknown")
        
        return hrtf, dimensions
        
    except Exception as e:
        print(f"Error loading SOFA file: {e}")
        import traceback
        traceback.print_exc()
        return None, None

def find_nearest_hrtf(hrtf, target_azimuth, target_elevation=0, target_distance=1.0):
    """Find the measurement closest to the target azimuth, elevation, and distance"""
    try:
        # Check if this is a CTF file (Common Transfer Function)
        # CTF files typically have only one measurement/position
        is_ctf = False
        if "CTF" in hrtf.GLOBAL_SOFAConventions or "CTF" in str(hrtf):
            is_ctf = True
            
        if is_ctf or (hasattr(hrtf, 'Data_IR') and hrtf.Data_IR.shape[0] == 1 and hrtf.Data_IR.shape[1] == 1):
            print(f"Detected CTF file - using the single filter for azimuth {target_azimuth}°")
            # For CTF, there's only one measurement so return index 0
            # Try to get the position if available, otherwise use default
            if hasattr(hrtf, 'SourcePosition') and len(hrtf.SourcePosition) > 0:
                nearest_pos = hrtf.SourcePosition[0] if len(hrtf.SourcePosition.shape) == 1 else hrtf.SourcePosition[0,:]
            else:
                nearest_pos = np.array([0.0, 0.0, 0.0])  # Default position
            return 0, nearest_pos
    
        # Get source positions for regular HRTF files
        source_positions = np.array(hrtf.SourcePosition)
        
        # Handle different position array shapes
        if source_positions.shape[0] == 3 and len(source_positions.shape) == 2:
            source_positions = source_positions.T
        
        # Check if positions are in degrees or radians
        position_type = hrtf.SourcePosition_Type
        max_azimuth = np.max(np.abs(source_positions[:, 0]))
        positions_in_degrees = max_azimuth > 6.28
        
        if positions_in_degrees:
            # Both source positions and target are in degrees
            az_compare = target_azimuth
            el_compare = target_elevation
        else:
            # Source positions in radians, convert target to radians
            az_compare = np.deg2rad(target_azimuth)
            el_compare = np.deg2rad(target_elevation)
        
        # Handle negative azimuths if data is in 0-360 format
        if positions_in_degrees and target_azimuth < 0:
            # Check if we have negative azimuths in the data
            has_negative_az = np.any(source_positions[:, 0] < 0)
            if not has_negative_az:
                # Convert target from -180 to 180 format to 0 to 360 format
                az_compare = (target_azimuth + 360) % 360
        
        # Create distance metric focusing on azimuth and elevation
        if position_type == 'spherical':
            dists = np.sqrt(
                2 * (source_positions[:, 0] - az_compare)**2 + 
                (source_positions[:, 1] - el_compare)**2
            )
        else:
            # For Cartesian or unknown position type
            dists = np.abs(source_positions[:, 0] - az_compare)
                
        # Find nearest position
        nearest_idx = np.argmin(dists)
        nearest_pos = source_positions[nearest_idx]
        
        return nearest_idx, nearest_pos
        
    except Exception as e:
        print(f"Error finding nearest HRTF: {e}")
        import traceback
        traceback.print_exc()
        return 0, None

def extract_hrtf_data(hrtf, idx):
    """Extract HRTF impulse responses for a given index"""
    hrtf_data = hrtf.Data_IR
    
    # Check if this is a CTF file (Common Transfer Function)
    # CTF files often have shape (1, 1, N) or similar with only one filter
    is_ctf = False
    if "CTF" in hrtf.GLOBAL_SOFAConventions or "CTF" in str(hrtf):
        is_ctf = True
    
    # Handle CTF files (apply same filter to both ears)
    if is_ctf or hrtf_data.shape[0] == 1 and hrtf_data.shape[1] == 1:
        print("Detected CTF file - using same filter for both ears")
        try:
            # Get the single filter
            common_filter = hrtf_data[0, 0, :]
            return common_filter, common_filter
        except Exception as e:
            print(f"Error extracting CTF data: {e}")
            # Return simple impulse responses as fallback
            dummy_ir = np.zeros(256)
            dummy_ir[0] = 1.0
            return dummy_ir, dummy_ir
    
    # Regular HRTF files with separate left/right filters
    # Check dimensions and extract
    if hrtf_data.shape[0] == 2 and hrtf_data.shape[1] > 2:
        # Shape is (R, M, N)
        hrtf_left = hrtf_data[0, idx, :]
        hrtf_right = hrtf_data[1, idx, :]
    else:
        # Shape is (M, R, N)
        try:
            hrtf_left = hrtf_data[idx, 0, :]
            hrtf_right = hrtf_data[idx, 1, :]
        except:
            # Try other dimension orderings if the above fails
            try:
                if len(hrtf_data.shape) == 4:  # If 4D array
                    hrtf_left = hrtf_data[idx, 0, 0, :]
                    hrtf_right = hrtf_data[idx, 1, 0, :]
                else:
                    raise ValueError(f"Unexpected HRTF data shape: {hrtf_data.shape}")
            except Exception as e:
                print(f"Error extracting HRTF data: {e}")
                # Return empty IRs as fallback
                dummy_ir = np.zeros(256)
                dummy_ir[0] = 1.0
                return dummy_ir, dummy_ir
    
    return hrtf_left, hrtf_right

def create_approaching_movement_physics(hrtf, source_signal, azimuth, verbose=True):
    """
    Create audio where the sound approaches from a fixed azimuth using the EPFL LEMA
    physically-based approach for distance effects
    """
    # Calculate total samples
    total_samples = int(DURATION * SAMPLE_RATE)
    
    # Use HRTF for directional cues - find closest matching HRTF for our azimuth
    nearest_idx, nearest_pos = find_nearest_hrtf(hrtf, azimuth, CONSTANT_ELEVATION)
    hrtf_left, hrtf_right = extract_hrtf_data(hrtf, nearest_idx)
    
    if verbose:
        if nearest_pos is not None:
            try:
                # For spherical coordinates
                if hasattr(hrtf, 'SourcePosition_Type') and hrtf.SourcePosition_Type == 'spherical':
                    # Check if in degrees or radians
                    if isinstance(nearest_pos, np.ndarray) and len(nearest_pos) > 0:
                        if np.max(np.abs(nearest_pos[0])) > 6.28:
                            pos_str = f"({nearest_pos[0]:.1f}°, {nearest_pos[1]:.1f}°)"
                        else:
                            pos_str = f"({np.rad2deg(nearest_pos[0]):.1f}°, {np.rad2deg(nearest_pos[1]):.1f}°)"
                    else:
                        pos_str = str(nearest_pos)
                else:
                    # For Cartesian or other formats
                    if isinstance(nearest_pos, np.ndarray) and len(nearest_pos) >= 3:
                        pos_str = f"[{nearest_pos[0]:.2f}, {nearest_pos[1]:.2f}, {nearest_pos[2]:.2f}]"
                    else:
                        pos_str = str(nearest_pos)
            except:
                pos_str = str(nearest_pos)
        else:
            pos_str = "unknown"
            
        print(f"Using HRTF measurement at position {pos_str}")
    
    # Apply the HRTF to the entire source signal for directional cues
    print("Applying HRTF filter for directional cues...")
    dir_left = signal.fftconvolve(source_signal, hrtf_left, mode='same')
    dir_right = signal.fftconvolve(source_signal, hrtf_right, mode='same')
    
    # Now apply the physics-based approach for distance effects
    
    # 1. Create source coordinates as it approaches for frontal approach
    # Set up the geometry (x, y) coordinates
    # Coordinates of the ears (left and right)
    coo_ear = np.array([[-HEAD_DIAMETER/2, 0], [HEAD_DIAMETER/2, 0]])
    
    # For frontal approach, x changes (decreases), y is constant
    x_start = -INITIAL_DISTANCE
    x_stop = -FINAL_DISTANCE
    y_value = 0
    
    # Create source coordinates for each sample
    time_points = np.linspace(0, DURATION, total_samples)
    x_positions = np.linspace(x_start, x_stop, total_samples)
    y_positions = np.full(total_samples, y_value)
    
    # Combine into source coordinates
    source_coordinates = np.column_stack((x_positions, y_positions))
    
    # 2. Calculate distances from the source to each ear at each time point
    # Distance from source to head center
    distance_to_head_center = np.sqrt(source_coordinates[:, 0]**2 + source_coordinates[:, 1]**2)
    
    # Calculate the direction of arrival angle for interaural time difference
    doa = np.zeros(total_samples)
    for i in range(total_samples):
        x_source = source_coordinates[i, 0]
        y_source = source_coordinates[i, 1]
        source_distance = np.sqrt(x_source**2 + y_source**2)
        
        if source_distance > 0:
            # Calculate angle in degrees
            angle_rad = np.arctan2(y_source, x_source)
            doa[i] = np.rad2deg(angle_rad)
        else:
            doa[i] = 0
    
    # Calculate distances from source to each ear
    # Using the spherical head model
    # Left ear
    dist_to_left_ear = np.zeros(total_samples)
    # Right ear
    dist_to_right_ear = np.zeros(total_samples)
    
    for i in range(total_samples):
        # Direct Euclidean distance from source to each ear
        x_source = source_coordinates[i, 0]
        y_source = source_coordinates[i, 1]
        
        # Distance to left ear
        dist_to_left_ear[i] = np.sqrt((x_source - coo_ear[0, 0])**2 + (y_source - coo_ear[0, 1])**2)
        
        # Distance to right ear
        dist_to_right_ear[i] = np.sqrt((x_source - coo_ear[1, 0])**2 + (y_source - coo_ear[1, 1])**2)
    
    # 3. Apply amplitude and delay based on distances
    # Calculate amplitude modulation using inverse square law with attenuation factor
    ref_level = 100  # Reference sound level
    min_distance = 0.001  # To avoid division by zero
    
    # Use the reference amplitude and attenuation factor
    ref_amplitude = 2e-5 * 10**((ref_level-3)/20)
    
    # Calculate amplitude scaling based on distance
    left_amp = np.zeros(total_samples)
    right_amp = np.zeros(total_samples)
    
    for i in range(total_samples):
        # Amplitude based on distance, with directional bias
        d_left = max(dist_to_left_ear[i], min_distance)
        d_right = max(dist_to_right_ear[i], min_distance)
        d_center = max(distance_to_head_center[i], min_distance)
        
        # Calculate amplitude attenuation
        # Directional component based on angle (adapted from MATLAB)
        a_L = 5/180
        b_L = -5
        loss_dB_L = a_L * doa[i] + b_L
        
        a_R = -5/180
        b_R = 0
        loss_dB_R = a_R * doa[i] + b_R
        
        # Combine direction and distance effects
        left_amp[i] = 2e-5 * 10**((loss_dB_L + DISTANCE_FACTOR * np.log10(1/d_center))/20)
        right_amp[i] = 2e-5 * 10**((loss_dB_R + DISTANCE_FACTOR * np.log10(1/d_center))/20)
    
    # Normalize and scale
    left_amp = left_amp / np.max(left_amp) * ref_amplitude
    right_amp = right_amp / np.max(right_amp) * ref_amplitude
    
    # 4. Calculate time delays based on distance differences
    # Convert distance difference to time delay in samples
    left_delay_samples = np.round(dist_to_left_ear / SPEED_OF_SOUND * SAMPLE_RATE).astype(int)
    right_delay_samples = np.round(dist_to_right_ear / SPEED_OF_SOUND * SAMPLE_RATE).astype(int)
    
    # Find maximum delay to ensure we have enough buffer
    max_delay = max(np.max(left_delay_samples), np.max(right_delay_samples))
    
    # 5. Create output buffer with extra space for delays
    output_signal = np.zeros((total_samples + max_delay, 2))
    
    # 6. Apply the amplitude and delay modulation to the directional signal
    for i in range(total_samples):
        # Apply amplitude scaling and add to output with delay
        output_signal[left_delay_samples[i] + i, 0] += dir_left[i] * left_amp[i]
        output_signal[right_delay_samples[i] + i, 1] += dir_right[i] * right_amp[i]
    
    # Ensure we have enough samples and trim if needed
    if len(output_signal) > total_samples:
        output_signal = output_signal[:total_samples]
    
    # 7. Apply a gentle fade out at the end to avoid any clicks
    fade_samples = int(0.1 * SAMPLE_RATE)
    if fade_samples < total_samples:
        fade_window = np.ones(total_samples)
        fade_window[-fade_samples:] = np.linspace(1.0, 0.0, fade_samples)
        
        output_signal[:, 0] *= fade_window
        output_signal[:, 1] *= fade_window
    
    # 8. Normalize to avoid clipping and apply volume control
    max_val = np.max(np.abs(output_signal))
    if max_val > 0:
        # First normalize to avoid clipping
        output_signal = output_signal / max_val * 0.9
        
        # Then apply the volume factor
        output_signal = output_signal * VOLUME_LEVEL
    
    # Create distance array for plotting
    distances = np.sqrt(x_positions**2 + y_positions**2)
    
    return output_signal, distances

def plot_distance_trajectory(distances, azimuth, noise_type, output_path):
    """Plot the trajectory of the approaching sound (distance over time)"""
    # Create figure
    plt.figure(figsize=(10, 6))
    
    # Define time axis (in seconds)
    time_axis = np.linspace(0, DURATION, len(distances))
    
    # Plot distance vs time
    plt.plot(time_axis, distances, 'b-', linewidth=3)
    
    # Scatter plot with color gradient representing distance
    scatter_main = plt.scatter(time_axis, distances, c=distances, cmap='viridis', s=80, alpha=0.7)
    
    # Add colorbar to explain the gradient colors
    cbar_main = plt.colorbar(scatter_main)
    cbar_main.set_label('Distance (meters)', fontsize=10)
    
    # Mark start and end points
    plt.scatter(time_axis[0], distances[0], color='green', s=120, marker='o', label="Start (Far)")
    plt.scatter(time_axis[-1], distances[-1], color='red', s=120, marker='o', label="End (Near)")
    
    # Add grid and labels
    plt.grid(True, linestyle='--', alpha=0.5)
    
    # Create noise type string with first letter capitalized
    noise_type_display = noise_type.capitalize()
    
    plt.title(f"Frontal Approaching Sound ({noise_type_display} Noise)", fontsize=14)
    plt.xlabel("Time (seconds)", fontsize=12)
    plt.ylabel("Distance from Listener (meters)", fontsize=12)
    plt.legend(fontsize=10)
    
    # Add annotations
    plt.annotate("Far", (time_axis[0], distances[0]), 
                xytext=(time_axis[0]-0.2, distances[0]+0.2),
                fontsize=12, fontweight='bold')
    plt.annotate("Near", (time_axis[-1], distances[-1]), 
                xytext=(time_axis[-1]-0.2, distances[-1]+0.2),
                fontsize=12, fontweight='bold')
    
    # Add top-down view of the scene - positioned higher to avoid interfering with axis
    ax_inset = plt.axes([0.15, 0.25, 0.3, 0.3])  # [left, bottom, width, height] - moved higher
    
    # Plot listener (center) and sound positions
    ax_inset.scatter([0], [0], color='black', s=200, marker='o', label="Listener")
    
    # Calculate positions along azimuth at different distances
    # We rotate by 90° to convert from azimuth to Cartesian
    x_positions = distances * np.cos(np.deg2rad(azimuth + 90))
    y_positions = distances * np.sin(np.deg2rad(azimuth + 90))
    
    # Plot trajectory points with color gradient
    scatter_inset = ax_inset.scatter(x_positions, y_positions, c=distances, cmap='viridis', s=50, alpha=0.7)
    ax_inset.plot(x_positions, y_positions, 'k-', alpha=0.5)
    
    # Add colorbar to inset plot
    cbar_inset = plt.colorbar(scatter_inset, ax=ax_inset, shrink=0.7)
    cbar_inset.set_label('Distance (m)', fontsize=8)
    
    # Mark start and end
    ax_inset.scatter(x_positions[0], y_positions[0], color='green', s=80, marker='o')
    ax_inset.scatter(x_positions[-1], y_positions[-1], color='red', s=80, marker='o')
    
    # Add an arrow to show direction of movement
    arrow_idx = len(distances) // 2
    ax_inset.arrow(x_positions[arrow_idx], y_positions[arrow_idx], 
                  (x_positions[arrow_idx+1] - x_positions[arrow_idx])*3, 
                  (y_positions[arrow_idx+1] - y_positions[arrow_idx])*3,
                  head_width=0.1, head_length=0.15, fc='red', ec='red')
    
    # Draw head
    head_circle = plt.Circle((0, 0), HEAD_DIAMETER/2, fill=False, color='blue', alpha=0.7)
    ax_inset.add_artist(head_circle)
    
    # Draw ears
    ax_inset.scatter([-HEAD_DIAMETER/2, HEAD_DIAMETER/2], [0, 0], color='blue', s=50, marker='o')
    
    # Set equal aspect ratio for inset
    ax_inset.set_aspect('equal')
    ax_inset.set_title("Top-Down View", fontsize=10)
    ax_inset.grid(True, linestyle='--', alpha=0.3)
    ax_inset.set_xlabel("X (meters)")
    ax_inset.set_ylabel("Y (meters)")
    
    # Add legend to bottom left of inset plot
    ax_inset.legend(loc='lower left', fontsize=8)
    
    # Set axis limits to show the geometry
    max_range = max(abs(INITIAL_DISTANCE), abs(FINAL_DISTANCE)) * 1.2
    ax_inset.set_xlim(-max_range, max_range)
    ax_inset.set_ylim(-max_range, max_range)
    
    # Save figure
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Saved trajectory plot to: {output_path}")

def main():
    """Generate frontal approaching sounds using different noise types with Fabian SOFA models"""
    if not SOFA_AVAILABLE:
        print("ERROR: sofar library is required. Please install it with:")
        print("pip install sofar")
        return
    
    # Process each SOFA file
    valid_files = []
    
    for sofa_path in sofa_files:
        if not os.path.exists(sofa_path):
            print(f"File not found: {sofa_path}")
            continue
        
        valid_files.append(sofa_path)
        file_name = os.path.basename(sofa_path).replace('.sofa', '')
        print(f"\nProcessing {file_name}...")
        
        try:
            # Load SOFA file
            hrtf, dimensions = load_sofa_file(sofa_path)
            
            if hrtf is None:
                print(f"Failed to load {file_name}, skipping.")
                continue
            
            # Generate each noise type
            for noise_type in NOISE_TYPES:
                print(f"\nGenerating approaching sound with {noise_type} noise...")
                
                # Generate source signal based on noise type
                source_signal = generate_noise(noise_type, DURATION, SAMPLE_RATE)
                
                # Create approaching movement with physics-based model
                output_signal, distances = create_approaching_movement_physics(hrtf, source_signal, AZIMUTH_ANGLE)
                
                # Save audio file with descriptive name
                output_filename = f"frontal_looming_{noise_type}_noise.wav"
                output_path = os.path.join(output_dir, output_filename)
                sf.write(output_path, output_signal, SAMPLE_RATE)
                print(f"Saved audio to: {output_path}")
                
                # Create trajectory plot
                plot_path = os.path.join(output_dir, f"frontal_looming_{noise_type}_noise_trajectory.png")
                plot_distance_trajectory(distances, AZIMUTH_ANGLE, noise_type, plot_path)
            
        except Exception as e:
            print(f"Error processing {file_name}: {e}")
            import traceback
            traceback.print_exc()
    
    print("\nFrontal looming sound generation complete!")
    print(f"Total SOFA models processed: {len(valid_files)}")
    print(f"All files saved to: {output_dir}")
    
    # Summarize settings used
    print("\nSettings used:")
    print(f"  Initial distance: {INITIAL_DISTANCE}m")
    print(f"  Final distance: {FINAL_DISTANCE}m")
    print(f"  Motion duration: {DURATION}s")
    print(f"  Azimuth: {AZIMUTH_ANGLE}° (frontal approach)")
    print(f"  Elevation: {CONSTANT_ELEVATION}°")
    print(f"  Head diameter: {HEAD_DIAMETER}m (natural human head size)")
    print(f"  Distance attenuation factor: {DISTANCE_FACTOR}")
    print(f"  Volume level: {VOLUME_LEVEL:.1f} (0.0-1.0)")
    print(f"  Noise types generated: {', '.join(NOISE_TYPES)}")
    print(f"  HRTF model used: FABIAN_HRIR_measured_HATO_0 (for natural binaural cues)")

if __name__ == "__main__":
    main()


Processing FABIAN_HRIR_measured_HATO_0...
Loading SOFA file: FABIAN_HRIR_measured_HATO_0.sofa
SOFA convention: SimpleFreeFieldHRIR v1.0
M = 11950
R = 2
N = 256

Generating approaching sound with pink noise...
Using HRTF measurement at position (0.0°, 0.0°)
Applying HRTF filter for directional cues...
Saved audio to: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\frontal_looming_pink_noise.wav
Saved trajectory plot to: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\frontal_looming_pink_noise_trajectory.png

Generating approaching sound with white noise...
Using HRTF measurement at position (0.0°, 0.0°)
Applying HRTF filter for directional cues...
Saved audio to: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\frontal_looming_white_noise.wav
Saved trajectory plot to: C:\Users\cogpsy-vrlab\Documents\GitHub\BreathingSpace\Level0_StimulusFiles\frontal_looming_white_noise_trajectory.png

Generating approachin

In [1]:
"""
Simplified Frontal Looming Sound Generator

This script creates binaural audio files simulating a sound source approaching the listener
from directly in front (0° azimuth), using HRTF for spatial filtering and a physically-based
amplitude envelope for distance perception.

Key features:
- Pure frontal approach (no ITD, no ILD)
- Physically-based inverse square law for amplitude
- Multiple noise types supported
- Simple, transparent processing chain
"""
import numpy as np
import scipy.signal as signal
import soundfile as sf
import matplotlib.pyplot as plt
import os

# Check for required libraries
try:
    import sofar
    SOFA_AVAILABLE = True
except ImportError:
    print("Warning: sofar library not found. Please install with: pip install sofar --break-system-packages")
    SOFA_AVAILABLE = False

#############################################################
# CONTROL PARAMETERS
#############################################################

# Spatial parameters
INITIAL_DISTANCE = 2.0      # Starting distance in meters
FINAL_DISTANCE = 0.5        # Final distance in meters
AZIMUTH_ANGLE = 0           # Frontal approach (0 degrees)
ELEVATION_ANGLE = 0         # At ear level

# Audio parameters
DURATION = 3.0              # Duration of approach in seconds
SAMPLE_RATE = 44100         # Audio sample rate in Hz
VOLUME_LEVEL = 0.7          # Output volume (0.0 to 1.0)

# Distance attenuation using inverse square law
# Intensity = 1 / distance^2
# Sound pressure level (amplitude) = 1 / distance
USE_INVERSE_SQUARE_LAW = True

# Noise types to generate (will be numbered sequentially)
NOISE_TYPES = ['pink', 'blue', 'white', 'brown']

# SOFA file path
SOFA_FILE = r"C:\Users\cogpsy-vrlab\Downloads\HRIRs_neutral_head_orientation_v4\SOFA\FABIAN_HRIR_measured_HATO_0.sofa"

# Output directory
OUTPUT_DIR = r"G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\LoomingStimuli"

#############################################################
# NOISE GENERATION
#############################################################

def generate_noise(noise_type, duration, fs):
    """Generate different types of noise with consistent spectral characteristics"""
    num_samples = int(duration * fs)
    
    # Use fixed seed for reproducibility
    np.random.seed(42)
    white_noise = np.random.randn(num_samples)
    
    if noise_type == 'white':
        noise = white_noise
        
    elif noise_type == 'pink':
        # Pink noise (1/f spectrum) using Voss-McCartney algorithm approximation
        b = np.array([0.049922035, -0.095993537, 0.050612699, -0.004408786])
        a = np.array([1.0, -2.494956002, 2.017265875, -0.522189400])
        noise = signal.lfilter(b, a, white_noise)
        
    elif noise_type == 'brown':
        # Brown/Brownian noise (1/f^2 spectrum)
        b = [1.0]
        a = [1.0, -0.98]
        noise = signal.lfilter(b, a, white_noise)
        
    elif noise_type == 'blue':
        # Blue noise (f spectrum, high-frequency emphasis)
        b = [1.0, -1.0]
        a = [1.0, -0.98]
        noise = signal.lfilter(b, a, white_noise)
        
    else:
        print(f"Warning: Noise type '{noise_type}' not recognized. Using white noise.")
        noise = white_noise
    
    # Normalize to prevent clipping
    noise = noise / np.max(np.abs(noise)) * 0.8
    
    return noise

#############################################################
# SOFA HRTF HANDLING
#############################################################

def load_sofa_file(sofa_path):
    """Load HRTF data from SOFA file"""
    try:
        print(f"Loading SOFA file: {os.path.basename(sofa_path)}")
        hrtf = sofar.read_sofa(sofa_path)
        
        print(f"SOFA Convention: {hrtf.GLOBAL_SOFAConventions} v{hrtf.GLOBAL_SOFAConventionsVersion}")
        
        # Get basic dimensions
        if hasattr(hrtf, 'Data_IR'):
            print(f"HRTF shape: {hrtf.Data_IR.shape}")
        
        return hrtf
        
    except Exception as e:
        print(f"Error loading SOFA file: {e}")
        import traceback
        traceback.print_exc()
        return None

def find_frontal_hrtf(hrtf, target_azimuth=0, target_elevation=0):
    """Find the HRTF measurement closest to frontal position (0°, 0°)"""
    try:
        # Check if this is a CTF (Common Transfer Function) file
        is_ctf = "CTF" in str(hrtf.GLOBAL_SOFAConventions)
        
        if is_ctf or (hasattr(hrtf, 'Data_IR') and hrtf.Data_IR.shape[0] <= 2 and hrtf.Data_IR.shape[1] == 1):
            print("Detected CTF file - using single filter")
            return 0, np.array([0.0, 0.0, 0.0])
        
        # For regular HRTF files, find closest to target position
        source_positions = np.array(hrtf.SourcePosition)
        
        # Handle different array shapes
        if source_positions.shape[0] == 3 and len(source_positions.shape) == 2:
            source_positions = source_positions.T
        
        # Determine if positions are in degrees or radians
        max_azimuth = np.max(np.abs(source_positions[:, 0]))
        positions_in_degrees = max_azimuth > 6.28
        
        if positions_in_degrees:
            az_target = target_azimuth
            el_target = target_elevation
        else:
            az_target = np.deg2rad(target_azimuth)
            el_target = np.deg2rad(target_elevation)
        
        # Find nearest position (prioritize azimuth match)
        distances = np.sqrt(
            (source_positions[:, 0] - az_target)**2 + 
            (source_positions[:, 1] - el_target)**2
        )
        
        nearest_idx = np.argmin(distances)
        nearest_pos = source_positions[nearest_idx]
        
        # Display the matched position
        if positions_in_degrees:
            print(f"Using HRTF at position: Az={nearest_pos[0]:.1f}°, El={nearest_pos[1]:.1f}°")
        else:
            print(f"Using HRTF at position: Az={np.rad2deg(nearest_pos[0]):.1f}°, El={np.rad2deg(nearest_pos[1]):.1f}°")
        
        return nearest_idx, nearest_pos
        
    except Exception as e:
        print(f"Error finding frontal HRTF: {e}")
        import traceback
        traceback.print_exc()
        return 0, None

def extract_hrtf_filters(hrtf, idx):
    """Extract left and right ear HRTF impulse responses"""
    hrtf_data = hrtf.Data_IR
    
    # Check for CTF (same filter for both ears)
    is_ctf = "CTF" in str(hrtf.GLOBAL_SOFAConventions)
    
    if is_ctf or (hrtf_data.shape[0] <= 2 and hrtf_data.shape[1] == 1):
        print("Using same filter for both ears (CTF)")
        try:
            common_filter = hrtf_data[0, 0, :]
            return common_filter, common_filter
        except:
            # Fallback: simple impulse
            dummy_ir = np.zeros(256)
            dummy_ir[0] = 1.0
            return dummy_ir, dummy_ir
    
    # Regular HRTF with separate left/right filters
    try:
        # Try common dimension ordering
        if hrtf_data.shape[0] == 2 and hrtf_data.shape[1] > 2:
            # Shape: (R=2, M=positions, N=samples)
            hrtf_left = hrtf_data[0, idx, :]
            hrtf_right = hrtf_data[1, idx, :]
        else:
            # Shape: (M=positions, R=2, N=samples)
            hrtf_left = hrtf_data[idx, 0, :]
            hrtf_right = hrtf_data[idx, 1, :]
        
        return hrtf_left, hrtf_right
        
    except Exception as e:
        print(f"Error extracting HRTF filters: {e}")
        # Fallback
        dummy_ir = np.zeros(256)
        dummy_ir[0] = 1.0
        return dummy_ir, dummy_ir

#############################################################
# LOOMING STIMULUS GENERATION
#############################################################

def create_amplitude_envelope(duration, fs, initial_dist, final_dist, use_inverse_square=True):
    """
    Create amplitude envelope for approaching sound based on distance
    
    For frontal looming, the primary cue is increasing amplitude.
    Uses inverse square law: amplitude ∝ 1/distance
    (intensity ∝ 1/distance^2, but we perceive amplitude/pressure)
    """
    num_samples = int(duration * fs)
    
    # Create smooth distance trajectory (linear movement)
    distances = np.linspace(initial_dist, final_dist, num_samples)
    
    if use_inverse_square:
        # Inverse distance law (physically accurate)
        # Amplitude is inversely proportional to distance
        amplitudes = 1.0 / distances
    else:
        # Linear amplitude increase (less realistic but smoother)
        amplitudes = np.linspace(1.0/initial_dist, 1.0/final_dist, num_samples)
    
    # Normalize so maximum amplitude is 1.0
    amplitudes = amplitudes / np.max(amplitudes)
    
    # Apply gentle fade in/out to avoid clicks
    fade_samples = int(0.05 * fs)  # 50ms fade
    
    # Fade in at start
    fade_in = np.linspace(0, 1, fade_samples)
    amplitudes[:fade_samples] *= fade_in
    
    # Fade out at end
    fade_out = np.linspace(1, 0, fade_samples)
    amplitudes[-fade_samples:] *= fade_out
    
    return amplitudes, distances

def generate_looming_stimulus(hrtf, noise_type, verbose=True):
    """
    Generate a frontal looming stimulus
    
    Processing chain:
    1. Generate noise signal
    2. Apply frontal HRTF (spatial filtering)
    3. Apply distance-based amplitude envelope
    4. Keep both ears identical (no ITD/ILD for frontal approach)
    """
    if verbose:
        print(f"\nGenerating looming stimulus with {noise_type} noise...")
    
    # Step 1: Generate source noise
    source_signal = generate_noise(noise_type, DURATION, SAMPLE_RATE)
    
    # Step 2: Find and apply frontal HRTF
    nearest_idx, nearest_pos = find_frontal_hrtf(hrtf, AZIMUTH_ANGLE, ELEVATION_ANGLE)
    hrtf_left, hrtf_right = extract_hrtf_filters(hrtf, nearest_idx)
    
    if verbose:
        print("Applying frontal HRTF filters...")
    
    # Apply HRTF filters (spectral coloring from head/pinnae)
    filtered_left = signal.fftconvolve(source_signal, hrtf_left, mode='same')
    filtered_right = signal.fftconvolve(source_signal, hrtf_right, mode='same')
    
    # Step 3: Create distance-based amplitude envelope
    if verbose:
        print("Applying distance-based amplitude envelope...")
    
    amplitude_envelope, distances = create_amplitude_envelope(
        DURATION, SAMPLE_RATE, INITIAL_DISTANCE, FINAL_DISTANCE, USE_INVERSE_SQUARE_LAW
    )
    
    # Step 4: Apply amplitude envelope to both channels
    # For frontal approach, both ears receive identical amplitude changes
    output_left = filtered_left * amplitude_envelope
    output_right = filtered_right * amplitude_envelope
    
    # Combine into stereo signal
    output_signal = np.column_stack((output_left, output_right))
    
    # Normalize and apply volume control
    max_val = np.max(np.abs(output_signal))
    if max_val > 0:
        output_signal = output_signal / max_val * 0.95  # Avoid clipping
        output_signal = output_signal * VOLUME_LEVEL
    
    if verbose:
        print(f"Generated {len(output_signal)/SAMPLE_RATE:.2f}s looming stimulus")
    
    return output_signal, distances

#############################################################
# VISUALIZATION
#############################################################

def plot_looming_trajectory(distances, noise_type, output_path):
    """Create visualization of the looming trajectory"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    time_axis = np.linspace(0, DURATION, len(distances))
    
    # Left plot: Distance over time
    ax1.plot(time_axis, distances, 'b-', linewidth=2.5, label='Distance trajectory')
    ax1.fill_between(time_axis, distances, alpha=0.3)
    
    # Mark start and end
    ax1.scatter(time_axis[0], distances[0], color='green', s=150, 
                marker='o', label='Start (far)', zorder=5, edgecolor='darkgreen', linewidth=2)
    ax1.scatter(time_axis[-1], distances[-1], color='red', s=150, 
                marker='o', label='End (near)', zorder=5, edgecolor='darkred', linewidth=2)
    
    ax1.grid(True, linestyle='--', alpha=0.4)
    ax1.set_xlabel('Time (seconds)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Distance (meters)', fontsize=12, fontweight='bold')
    ax1.set_title(f'Frontal Looming: {noise_type.capitalize()} Noise', fontsize=13, fontweight='bold')
    ax1.legend(fontsize=10, loc='upper right')
    
    # Add velocity annotation
    velocity = (INITIAL_DISTANCE - FINAL_DISTANCE) / DURATION
    ax1.text(0.02, 0.98, f'Approach velocity: {velocity:.2f} m/s', 
             transform=ax1.transAxes, fontsize=10, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    # Right plot: Top-down view
    ax2.set_aspect('equal')
    
    # Draw listener (head and ears)
    head_circle = plt.Circle((0, 0), 0.09, fill=False, color='blue', linewidth=2, label='Listener (head)')
    ax2.add_artist(head_circle)
    ax2.scatter([0], [0], color='blue', s=100, marker='o', zorder=5)
    ax2.scatter([-0.09, 0.09], [0, 0], color='blue', s=80, marker='s', label='Ears')
    
    # Draw source trajectory (approaching from front)
    # Front is along positive y-axis
    source_x = np.zeros_like(distances)
    source_y = distances  # Approaching along y-axis
    
    # Plot trajectory
    scatter = ax2.scatter(source_x, source_y, c=time_axis, cmap='plasma', 
                         s=80, alpha=0.7, edgecolors='black', linewidth=0.5)
    ax2.plot(source_x, source_y, 'k--', alpha=0.3, linewidth=1)
    
    # Mark start and end positions
    ax2.scatter(source_x[0], source_y[0], color='green', s=200, marker='o', 
                edgecolor='darkgreen', linewidth=2, zorder=10, label='Start')
    ax2.scatter(source_x[-1], source_y[-1], color='red', s=200, marker='o', 
                edgecolor='darkred', linewidth=2, zorder=10, label='End')
    
    # Add arrow showing direction
    arrow_idx = len(distances) // 2
    ax2.annotate('', xy=(source_x[arrow_idx+10], source_y[arrow_idx+10]),
                xytext=(source_x[arrow_idx], source_y[arrow_idx]),
                arrowprops=dict(arrowstyle='->', lw=2, color='red'))
    
    # Colorbar for time
    cbar = plt.colorbar(scatter, ax=ax2)
    cbar.set_label('Time (seconds)', fontsize=10)
    
    ax2.grid(True, linestyle='--', alpha=0.3)
    ax2.set_xlabel('X position (meters)', fontsize=11)
    ax2.set_ylabel('Y position (meters)', fontsize=11)
    ax2.set_title('Top-Down View (Front = +Y)', fontsize=12, fontweight='bold')
    ax2.legend(fontsize=9, loc='upper left')
    
    # Set equal axis limits
    max_range = max(abs(INITIAL_DISTANCE), abs(FINAL_DISTANCE)) * 1.3
    ax2.set_xlim(-max_range, max_range)
    ax2.set_ylim(-0.2, max_range)
    
    # Add "FRONT" label
    ax2.text(0, max_range * 0.9, 'FRONT', fontsize=12, fontweight='bold', 
             ha='center', color='darkblue')
    
    plt.tight_layout()
    plt.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.close()
    
    print(f"Saved trajectory plot: {os.path.basename(output_path)}")

#############################################################
# MAIN EXECUTION
#############################################################

def main():
    """Generate frontal looming stimuli for all noise types"""
    
    print("=" * 70)
    print("FRONTAL LOOMING STIMULUS GENERATOR")
    print("=" * 70)
    
    # Check dependencies
    if not SOFA_AVAILABLE:
        print("\nERROR: sofar library is required.")
        print("Install with: pip install sofar --break-system-packages")
        return
    
    # Check SOFA file exists
    if not os.path.exists(SOFA_FILE):
        print(f"\nERROR: SOFA file not found: {SOFA_FILE}")
        return
    
    # Create output directory
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    print(f"\nOutput directory: {OUTPUT_DIR}")
    
    # Load HRTF
    print("\n" + "-" * 70)
    hrtf = load_sofa_file(SOFA_FILE)
    
    if hrtf is None:
        print("ERROR: Failed to load SOFA file")
        return
    
    # Generate stimuli for each noise type
    print("\n" + "-" * 70)
    print("GENERATING LOOMING STIMULI")
    print("-" * 70)
    
    generated_files = []
    
    for idx, noise_type in enumerate(NOISE_TYPES, start=1):
        print(f"\n[{idx}/{len(NOISE_TYPES)}] Processing {noise_type} noise...")
        
        try:
            # Generate looming stimulus
            output_signal, distances = generate_looming_stimulus(hrtf, noise_type, verbose=True)
            
            # Save audio file
            filename = f"Loom-{idx}-{noise_type}.wav"
            output_path = os.path.join(OUTPUT_DIR, filename)
            sf.write(output_path, output_signal, SAMPLE_RATE)
            print(f"✓ Saved: {filename}")
            generated_files.append(filename)
            
            # Create visualization
            plot_filename = f"Loom-{idx}-{noise_type}_trajectory.png"
            plot_path = os.path.join(OUTPUT_DIR, plot_filename)
            plot_looming_trajectory(distances, noise_type, plot_path)
            
        except Exception as e:
            print(f"✗ ERROR generating {noise_type} stimulus: {e}")
            import traceback
            traceback.print_exc()
    
    # Summary
    print("\n" + "=" * 70)
    print("GENERATION COMPLETE")
    print("=" * 70)
    print(f"\nGenerated {len(generated_files)} looming stimuli:")
    for filename in generated_files:
        print(f"  • {filename}")
    
    print(f"\nParameters used:")
    print(f"  • Initial distance: {INITIAL_DISTANCE} m")
    print(f"  • Final distance: {FINAL_DISTANCE} m")
    print(f"  • Duration: {DURATION} s")
    print(f"  • Approach velocity: {(INITIAL_DISTANCE - FINAL_DISTANCE) / DURATION:.2f} m/s")
    print(f"  • Azimuth: {AZIMUTH_ANGLE}° (frontal)")
    print(f"  • Elevation: {ELEVATION_ANGLE}° (ear level)")
    print(f"  • Sample rate: {SAMPLE_RATE} Hz")
    print(f"  • Volume level: {VOLUME_LEVEL:.1f}")
    print(f"  • Distance model: {'Inverse square law' if USE_INVERSE_SQUARE_LAW else 'Linear'}")
    print(f"  • HRTF model: FABIAN (neutral head orientation)")
    
    print(f"\nAll files saved to:")
    print(f"  {OUTPUT_DIR}")
    print("\n" + "=" * 70)

if __name__ == "__main__":
    main()

FRONTAL LOOMING STIMULUS GENERATOR

Output directory: G:\My Drive\Project2_PeripersonalSpace\ExperimentFiles\0_Stimuli\LoomingStimuli

----------------------------------------------------------------------
Loading SOFA file: FABIAN_HRIR_measured_HATO_0.sofa
SOFA Convention: SimpleFreeFieldHRIR v1.0
HRTF shape: (11950, 2, 256)

----------------------------------------------------------------------
GENERATING LOOMING STIMULI
----------------------------------------------------------------------

[1/4] Processing pink noise...

Generating looming stimulus with pink noise...
Using HRTF at position: Az=0.0°, El=0.0°
Applying frontal HRTF filters...
Applying distance-based amplitude envelope...
Generated 3.00s looming stimulus
✓ Saved: Loom-1-pink.wav
Saved trajectory plot: Loom-1-pink_trajectory.png

[2/4] Processing blue noise...

Generating looming stimulus with blue noise...
Using HRTF at position: Az=0.0°, El=0.0°
Applying frontal HRTF filters...
Applying distance-based amplitude envelo

In [1]:
import sys
import subprocess

print("=" * 70)
print("INSTALLING DEPENDENCIES IN PPS ENVIRONMENT")
print("=" * 70)
print(f"\nPython: {sys.executable}\n")

# Install packages
packages = {
    'numpy<2.0': 'NumPy',
    'scipy': 'SciPy',
    'matplotlib': 'Matplotlib',
    'soundfile': 'SoundFile',
    'sofar': 'SOFAR',
    'ipykernel': 'IPython Kernel'
}

print("Installing packages...\n")

for package, description in packages.items():
    print(f"📦 {description}...", end=" ")
    try:
        subprocess.check_call(
            [sys.executable, "-m", "pip", "install", package],
            stdout=subprocess.DEVNULL,
            stderr=subprocess.DEVNULL
        )
        print("✓")
    except:
        print("✗")

print("\n" + "=" * 70)
print("VERIFYING INSTALLATIONS")
print("=" * 70)

# Verify
import importlib

checks = ['numpy', 'scipy', 'scipy.signal', 'matplotlib', 'soundfile', 'sofar']
all_good = True

for module_name in checks:
    try:
        module = importlib.import_module(module_name)
        version = getattr(module, '__version__', 'OK')
        print(f"✓ {module_name:20s} [{version}]")
    except:
        print(f"✗ {module_name:20s} [FAILED]")
        all_good = False

# Check NumPy version
import numpy as np
if int(np.__version__.split('.')[0]) >= 2:
    print(f"\n⚠️  NumPy {np.__version__} - Need to restart kernel!")
    all_good = False
else:
    print(f"\n✓ NumPy {np.__version__} - Compatible!")

print("=" * 70)

if all_good:
    print("\n🎉 ALL PACKAGES READY! You can run the looming generator now.")
else:
    print("\n⚠️ Restart kernel (Kernel → Restart) then re-run to verify.")

INSTALLING DEPENDENCIES IN PPS ENVIRONMENT

Python: C:\Users\cogpsy-vrlab\anaconda3\envs\notebookStimulus\python.exe

Installing packages...

📦 NumPy... ✗
📦 SciPy... ✓
📦 Matplotlib... ✓
📦 SoundFile... ✓
📦 SOFAR... ✓
📦 IPython Kernel... ✓

VERIFYING INSTALLATIONS
✓ numpy                [2.3.5]
✓ scipy                [1.16.3]
✓ scipy.signal         [OK]
✓ matplotlib           [3.10.7]
✓ soundfile            [0.13.1]
✓ sofar                [1.2.2]

⚠️  NumPy 2.3.5 - Need to restart kernel!

⚠️ Restart kernel (Kernel → Restart) then re-run to verify.


In [ ]:
import sys
import subprocess

print("=" * 70)
print("FORCE DOWNGRADING NUMPY")
print("=" * 70)

print("\nDowngrading NumPy to compatible version...")
try:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "numpy==1.26.4", "--force-reinstall"],
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE
    )
    print("✓ NumPy downgrade complete")
except Exception as e:
    print(f"✗ Error: {e}")

print("\n" + "=" * 70)
print("⚠️  IMPORTANT: RESTART KERNEL NOW!")
print("=" * 70)
print("\nSteps:")
print("1. Click: Kernel → Restart Kernel")
print("2. Run verification cell below after restart")
print("=" * 70)

FORCE DOWNGRADING NUMPY

Downgrading NumPy to compatible version...
